In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict
import re
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rating: int

In [4]:
def create_outline(state: BlogState) -> BlogState:
    prompt = f"Create a blog outline for the title: {state['title']}"
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:
    prompt = f"Write a blog post based on the following outline: {state['outline']}"
    blog = model.invoke(prompt).content
    state["content"] = blog
    return state

In [6]:
def rate_blog(state: BlogState) -> BlogState:
    prompt = f"Rate the following blog post on a scale of 1 to 10. Return only the integer rating: {state['content']}"
    response = model.invoke(prompt).content
    rating_text = response[0]["text"] if isinstance(response, list) else str(response)
    rating = int(re.search(r"\b(10|[1-9])\b", rating_text).group(1))
    return {
        "title": state["title"],
        "outline": state["outline"],
        "content": state["content"],
        "rating": rating,
    }

In [7]:
graph = StateGraph(BlogState)
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_node("rate_blog", rate_blog)
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "rate_blog")
graph.add_edge("rate_blog", END)
workflow = graph.compile()

In [8]:
initial_state = {"title": "The Future of AI in Healthcare"}
final_state = workflow.invoke(initial_state)
# print(final_state["outline"])
# print(final_state["content"])
print(final_state["rating"])

9
